In [14]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [15]:
# Подготовка данных (пример)
X1 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_2_1.csv')
S1 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_2_1.csv')
y1 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_wheel_2_1.csv')

In [16]:
# Подготовка данных (пример)
X2 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_2_2.csv')
S2 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_2_2.csv')
y2 = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_wheel_2_2.csv')

In [17]:
X = pd.concat([X1, X2], ignore_index=True)
S = pd.concat([S1, S2], ignore_index=True)
y = pd.concat([y1, y2], ignore_index=True)

In [18]:
X.to_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_3.csv', index=False)

In [19]:
S = S / 170
S.to_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_3.csv', index=False)

In [20]:
y.to_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_3.csv', index=False)

In [5]:
print(len(X))
print(len(S))
print(len(y))

30650
30650
30650


In [6]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda') // 170
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [7]:
dataset = TensorDataset(X_train, S_train, y_tensor)
data_loader = DataLoader(dataset, batch_size=128, shuffle=True)

In [8]:
class FeedforwardNet(nn.Module):
    def __init__(self, input_size=12288, hidden_size_1=256, hidden_size_2=128, hidden_size_3=64, 
                 hidden_size_4=164, output_size=2):
        super(FeedforwardNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.fc3 = nn.Linear(hidden_size_2 + 1, hidden_size_3)
        self.fc4 = nn.Linear(hidden_size_3, output_size)
        # self.fc5 = nn.Linear(hidden_size_4, output_size)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, image, speed):
        out = self.fc1(image)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(torch.cat((out, speed), dim=1))
        out = self.relu(out)
        out = self.fc4(out)
        # out = self.relu(out)
        # out = self.fc5(out)
        # out = self.relu(out)
        out = self.tanh(out)
        return out

In [9]:
model = FeedforwardNet().cuda()

In [10]:
# Определение функции потерь и оптимизатора
criterion = nn.MultiLabelSoftMarginLoss()  # Функция потерь для регрессионной задачи
optimizer = optim.Adam(model.parameters())  # Оптимизатор Adam


# Обучение модели
num_epochs = 15
for epoch in range(num_epochs):
    for batch_x, batch_s, batch_y in data_loader:  # Итерация по батчам данных
        optimizer.zero_grad()  # Обнуление градиентов
        outputs = model(batch_x, batch_s)  # Передача входных данных через модель
        loss = criterion(outputs, batch_y)  # Вычисление потерь

        loss.backward()  # Обратное распространение ошибки
        optimizer.step()  # Обновление весов модели

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

Epoch [1/15], Loss: 0.316980928
Epoch [2/15], Loss: 0.319885135
Epoch [3/15], Loss: 0.315273821
Epoch [4/15], Loss: 0.322708517
Epoch [5/15], Loss: 0.308636695
Epoch [6/15], Loss: 0.321412295
Epoch [7/15], Loss: 0.319328696
Epoch [8/15], Loss: 0.315928608


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001DB6985A1D0>>
Traceback (most recent call last):
  File "C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\ipykernel\ipkernel.py", line 770, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
num_epochs = 1
for epoch in range(num_epochs):
    for batch_x, batch_s, batch_y in data_loader:  # Итерация по батчам данных
        optimizer.zero_grad()  # Обнуление градиентов
        outputs = model(batch_x, batch_s)  # Передача входных данных через модель
        loss = criterion(outputs, batch_y)  # Вычисление потерь

        loss.backward()  # Обратное распространение ошибки
        optimizer.step()  # Обновление весов модели

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

In [ ]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_forward_5.pth')